#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [111]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [112]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

In [113]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [114]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split ,cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [116]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [117]:
train_data = pd.read_csv('data01_train.csv')

In [118]:
test_data = pd.read_csv('data01_test.csv')

#### 2) 기본 정보 조회

In [119]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [120]:
train_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [121]:
train_data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [122]:
train_data.drop('subject', axis=1, inplace=True)

In [123]:
train_data.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING


In [124]:
sum(train_data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [125]:
dynamic_activity = (train_data['Activity']=='WALKING') | (train_data['Activity']=='WALKING_UPSTAIRS') | (train_data['Activity']=='WALKING_DOWNSTAIRS')
train_data = train_data[dynamic_activity]

In [126]:
train_data

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,WALKING_DOWNSTAIRS
5,0.330708,0.007561,-0.061371,-0.215760,0.101075,0.072949,-0.269857,0.060060,0.101298,-0.019263,...,-0.599906,-0.887024,-0.030645,-0.852091,-0.500195,0.306091,-0.552729,0.253885,0.291256,WALKING_UPSTAIRS
6,0.121465,-0.031902,-0.005196,-0.152198,-0.113104,-0.239423,-0.202401,-0.164698,-0.247099,0.114668,...,-0.371055,-0.775779,0.445206,-0.003487,-0.940185,0.041387,-0.886603,0.173338,-0.005627,WALKING
12,0.303885,0.002768,-0.038613,-0.168656,0.190336,-0.140473,-0.205134,0.101144,-0.120572,-0.000818,...,-0.037750,-0.329728,-0.040030,0.257252,0.076091,-0.123425,-0.752882,0.266729,0.045692,WALKING
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5868,0.277709,-0.053919,-0.098746,-0.589970,-0.076626,-0.500837,-0.605474,-0.142798,-0.506696,-0.533485,...,0.029201,-0.268237,-0.076922,0.706620,-0.954969,-0.324779,-0.691663,0.153974,-0.196833,WALKING
5869,0.252496,-0.009773,-0.118293,-0.155168,-0.244513,-0.044524,-0.244367,-0.229509,-0.042199,0.291682,...,-0.475800,-0.752980,0.619246,-0.684483,0.924623,-0.353041,-0.775518,0.253218,0.013216,WALKING_UPSTAIRS
5873,0.264961,-0.034719,-0.088140,-0.368158,-0.074514,-0.229606,-0.394387,-0.082437,-0.243525,-0.261087,...,0.198624,-0.046157,0.037517,-0.903692,0.614933,-0.610918,-0.601604,0.335996,0.156518,WALKING
5877,0.191568,0.013328,-0.105174,-0.126969,-0.121729,-0.327480,-0.192523,-0.109923,-0.295286,0.078644,...,-0.622617,-0.879215,0.721718,0.623151,0.866858,-0.445660,-0.690278,0.303194,-0.044188,WALKING_UPSTAIRS


In [127]:
dynamic_activity = (test_data['Activity']=='WALKING') | (test_data['Activity']=='WALKING_UPSTAIRS') | (test_data['Activity']=='WALKING_DOWNSTAIRS')
test_data = test_data[dynamic_activity]

In [128]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [129]:
X_test = test_data.drop(['subject', 'Activity'], axis=1)
y_test = test_data.loc[:, 'Activity']

In [130]:
y_train.unique()

array(['WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'], dtype=object)

In [131]:
y_test.unique()

array(['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS'], dtype=object)

In [132]:
X_train

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,0.050888,-0.004012,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.013902,-0.157832,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758
5,0.330708,0.007561,-0.061371,-0.215760,0.101075,0.072949,-0.269857,0.060060,0.101298,-0.019263,...,0.114524,-0.599906,-0.887024,-0.030645,-0.852091,-0.500195,0.306091,-0.552729,0.253885,0.291256
6,0.121465,-0.031902,-0.005196,-0.152198,-0.113104,-0.239423,-0.202401,-0.164698,-0.247099,0.114668,...,-0.127764,-0.371055,-0.775779,0.445206,-0.003487,-0.940185,0.041387,-0.886603,0.173338,-0.005627
12,0.303885,0.002768,-0.038613,-0.168656,0.190336,-0.140473,-0.205134,0.101144,-0.120572,-0.000818,...,0.227840,-0.037750,-0.329728,-0.040030,0.257252,0.076091,-0.123425,-0.752882,0.266729,0.045692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5868,0.277709,-0.053919,-0.098746,-0.589970,-0.076626,-0.500837,-0.605474,-0.142798,-0.506696,-0.533485,...,-0.058903,0.029201,-0.268237,-0.076922,0.706620,-0.954969,-0.324779,-0.691663,0.153974,-0.196833
5869,0.252496,-0.009773,-0.118293,-0.155168,-0.244513,-0.044524,-0.244367,-0.229509,-0.042199,0.291682,...,0.263950,-0.475800,-0.752980,0.619246,-0.684483,0.924623,-0.353041,-0.775518,0.253218,0.013216
5873,0.264961,-0.034719,-0.088140,-0.368158,-0.074514,-0.229606,-0.394387,-0.082437,-0.243525,-0.261087,...,0.155018,0.198624,-0.046157,0.037517,-0.903692,0.614933,-0.610918,-0.601604,0.335996,0.156518
5877,0.191568,0.013328,-0.105174,-0.126969,-0.121729,-0.327480,-0.192523,-0.109923,-0.295286,0.078644,...,0.325461,-0.622617,-0.879215,0.721718,0.623151,0.866858,-0.445660,-0.690278,0.303194,-0.044188


### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [133]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [134]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [135]:
acc_df = pd.DataFrame({'모델': [],
                       '논문_정확도(교차검증)': [],
                       '논문_정확도(테스트)': [],
                       '논문_정확도(fi_교차검증)': [],
                       '논문_정확도(fi_테스트)': []
                      })
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)


In [136]:
kappa_df = pd.DataFrame({'모델': [],
                       '논문_kappa(교차검증)': [],
                       '논문_kappa(테스트)': [],
                      })
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)


In [137]:
kappa_scorer = make_scorer(cohen_kappa_score)

### (1) 알고리즘1 : RandomForest

In [138]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9716667261119236
평균 카파  : 0.9572924977148615


In [139]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9764890282131662
[[223   0   5]
 [  4 189   2]
 [  2   2 211]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.98      0.98       228
WALKING_DOWNSTAIRS       0.99      0.97      0.98       195
  WALKING_UPSTAIRS       0.97      0.98      0.97       215

          accuracy                           0.98       638
         macro avg       0.98      0.98      0.98       638
      weighted avg       0.98      0.98      0.98       638

Cohen's Kappa Score: 0.9646


In [140]:
joblib.dump(rf_model, 'rf_model_paper.pkl')

['rf_model_paper.pkl']

In [141]:
new_row = pd.DataFrame([{'모델': 'RandomForest', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [142]:
new_row = pd.DataFrame([{'모델': 'RandomForest', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [143]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,NaN,NaN


In [144]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.957292,0.964649


### (2) 알고리즘2 : LogisticRegressor

In [145]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
scores = cross_validate(lr_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9969761386738952
평균 카파  : 0.9954418036676896


In [146]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9984326018808778
[[228   0   0]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9976


In [147]:
joblib.dump(lr_model, 'lr_model_paper.pkl')

['lr_model_paper.pkl']

In [148]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [149]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [150]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,NaN,NaN
1,LogisticRegression,0.996976,0.998433,NaN,NaN


In [151]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.957292,0.964649
1,LogisticRegression,0.995442,0.997644


### (3) 알고리즘3 : SVM

In [152]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9984877126654064
평균 카파  : 0.9977206043335023


In [153]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9952978056426333
[[227   0   1]
 [  0 194   1]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       0.99      1.00      0.99       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9929


In [154]:
joblib.dump(svm_model, 'svm_model_paper.pkl')

['svm_model_paper.pkl']

In [155]:
new_row = pd.DataFrame([{'모델': 'SVM', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [156]:
new_row = pd.DataFrame([{'모델': 'SVM', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [157]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,NaN,NaN
1,LogisticRegression,0.996976,0.998433,NaN,NaN
2,SVM,0.998488,0.995298,NaN,NaN


In [158]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.957292,0.964649
1,LogisticRegression,0.995442,0.997644
2,SVM,0.997721,0.992932


### (4) 알고리즘4 : DecisionTreeClassifier

In [159]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9108485215964619
평균 카파  : 0.8657648109444972


In [160]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9122257053291536
[[208   9  11]
 [  3 185   7]
 [  6  20 189]]
                    precision    recall  f1-score   support

           WALKING       0.96      0.91      0.93       228
WALKING_DOWNSTAIRS       0.86      0.95      0.90       195
  WALKING_UPSTAIRS       0.91      0.88      0.90       215

          accuracy                           0.91       638
         macro avg       0.91      0.91      0.91       638
      weighted avg       0.91      0.91      0.91       638

Cohen's Kappa Score: 0.8683


In [161]:
joblib.dump(dt_model, 'dt_model_paper.pkl')

['dt_model_paper.pkl']

In [162]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [163]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [164]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,NaN,NaN
1,LogisticRegression,0.996976,0.998433,NaN,NaN
2,SVM,0.998488,0.995298,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,NaN,NaN


In [165]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.957292,0.964649
1,LogisticRegression,0.995442,0.997644
2,SVM,0.997721,0.992932
3,DecisionTreeClassifier,0.865765,0.868324


In [166]:
acc_df.to_csv('모델 정확도(논문 기준).csv', index=False, encoding='utf-8')

In [167]:
kappa_df.to_csv('모델 kappa(논문 기준).csv', index=False, encoding='utf-8')

### (5) 알고리즘5 : XGBoost

In [168]:
# le = LabelEncoder()
# y_xgb = le.fit_transform(y)

In [169]:
# X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb = train_test_split(X, y_xgb, test_size=0.3, random_state=42)

In [170]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train_xgb, y_train_xgb)

# y_pred = XGB_model.predict(X_val_xgb)
# y_pred = le.inverse_transform(y_pred)

# y_val = le.inverse_transform(y_val_xgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [171]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_val, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [172]:
# acc_df

### (5) 알고리즘5 : LightGBM

In [173]:
# X_lgb = X.copy()
# X_lgb.columns = [f'feature_{i}' for i in range(X.shape[1])]

In [174]:
# le = LabelEncoder()
# y_lgb = le.fit_transform(y)

In [175]:
# X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb = train_test_split(X_lgb, y_lgb, test_size=0.3, random_state=42)

In [176]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_val_lgb)
# y_pred = le.inverse_transform(y_pred)

# y_val= le.inverse_transform(y_val_lgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [177]:
# y_pred = model.predict(x_val)
# y_pred = np.argmax(y_pred, axis=1)

In [178]:
# y_val = le.inverse_transform(y_val)
# y_pred = le.inverse_transform(y_pred)
# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

# =====특성 중요도=======================


## 방법1. 랜덤포레스트에서 추출한 feature_importance_ 사용

In [179]:
acc_df = pd.read_csv('모델 정확도(논문 기준).csv')
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,NaN,NaN
1,LogisticRegression,0.996976,0.998433,NaN,NaN
2,SVM,0.998488,0.995298,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,NaN,NaN


In [180]:
import joblib
rf_model = joblib.load('rf_model_paper.pkl')

In [181]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,"tGravityAcc-arCoeff()-Z,2",0.034472
1,tBodyAccMag-std(),0.034371
2,fBodyAccMag-energy(),0.031183
3,"tGravityAcc-arCoeff()-Y,1",0.026519
4,fBodyAccMag-mad(),0.024388
...,...,...
556,fBodyBodyAccJerkMag-meanFreq(),0.000000
557,tBodyGyroJerk-mean()-Y,0.000000
558,tGravityAcc-entropy()-Y,0.000000
559,"fBodyGyro-bandsEnergy()-49,56.2",0.000000


In [182]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [183]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [184]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 200


In [185]:
selected_cum_importance = RFmodel_fi_df.loc[:len(selected_features), 'feature_importance'].sum()
print(f"상위 {len(selected_features)+1}개 변수의 누적 중요도: {selected_cum_importance:.4f}")

상위 201개 변수의 누적 중요도: 0.9503


In [186]:
X_train_fi = train_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]
y_train_fi = train_data.loc[:, 'Activity']

In [187]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [188]:
X_test_fi = test_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]

### 랜덤 포레스트

In [189]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9720455112886544
평균 카파  : 0.9578686681500137


In [190]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train_fi, y_train_fi)
y_pred = rf_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.987460815047022
[[224   0   4]
 [  1 193   1]
 [  1   1 213]]
                    precision    recall  f1-score   support

           WALKING       0.99      0.98      0.99       228
WALKING_DOWNSTAIRS       0.99      0.99      0.99       195
  WALKING_UPSTAIRS       0.98      0.99      0.98       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9812


In [191]:
rf_model = joblib.dump(rf_model, 'rf_model_paper_fi.pkl')

In [192]:
acc_df.loc[0, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [193]:
kappa_df.loc[0, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [194]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461
1,LogisticRegression,0.996976,0.998433,NaN,NaN
2,SVM,0.998488,0.995298,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,NaN,NaN


In [195]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153
1,LogisticRegression,0.995442,0.997644,NaN,NaN
2,SVM,0.997721,0.992932,NaN,NaN
3,DecisionTreeClassifier,0.865765,0.868324,NaN,NaN


### LinearRegression

In [196]:
# 랜덤 포레스트에서 추출한 중요도 상위 특성 활용
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
scores = cross_validate(lr_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9969768520169776
평균 카파  : 0.9954433251480553


In [197]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_fi, y_train_fi)
y_pred = lr_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9952978056426333
[[228   0   0]
 [  1 193   1]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       0.99      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      0.99       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9929


In [198]:
joblib.dump(lr_model, 'lr_model_paper_fi.pkl')

['lr_model_paper_fi.pkl']

In [199]:
acc_df.loc[1, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [200]:
kappa_df.loc[1, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [201]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298
2,SVM,0.998488,0.995298,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,NaN,NaN


In [202]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931
2,SVM,0.997721,0.992932,NaN,NaN
3,DecisionTreeClassifier,0.865765,0.868324,NaN,NaN


### SVM

In [203]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9984891393515711
평균 카파  : 0.997722576879665


In [204]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_fi, y_train_fi)

y_pred = svm_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9968652037617555
[[228   0   0]
 [  0 194   1]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9953


In [205]:
joblib.dump(svm_model, 'svm_model_paper_fi.pkl')

['svm_model_paper_fi.pkl']

In [206]:
acc_df.loc[2, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [207]:
kappa_df.loc[2, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [208]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298
2,SVM,0.998488,0.995298,0.998489,0.996865
3,DecisionTreeClassifier,0.910849,0.912226,NaN,NaN


In [209]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931
2,SVM,0.997721,0.992932,0.997723,0.995288
3,DecisionTreeClassifier,0.865765,0.868324,NaN,NaN


### DecisionTreeClassifier

In [210]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9108478082533795
평균 카파  : 0.8657480558652176


In [211]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_fi, y_train_fi)

y_pred = dt_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9122257053291536
[[208   9  11]
 [  3 185   7]
 [  7  19 189]]
                    precision    recall  f1-score   support

           WALKING       0.95      0.91      0.93       228
WALKING_DOWNSTAIRS       0.87      0.95      0.91       195
  WALKING_UPSTAIRS       0.91      0.88      0.90       215

          accuracy                           0.91       638
         macro avg       0.91      0.91      0.91       638
      weighted avg       0.91      0.91      0.91       638

Cohen's Kappa Score: 0.8683


In [212]:
joblib.dump(dt_model, 'dt_model_paper_fi.pkl')

['dt_model_paper_fi.pkl']

In [213]:
acc_df.loc[3, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [214]:
kappa_df.loc[3, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [215]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298
2,SVM,0.998488,0.995298,0.998489,0.996865
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226


In [216]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931
2,SVM,0.997721,0.992932,0.997723,0.995288
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308


In [217]:
acc_df.to_csv('모델 정확도(논문 기준_fi).csv', index=False, encoding='utf-8-sig')

In [218]:
kappa_df.to_csv('모델 kappa(논문 기준_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [219]:
acc_df = pd.read_csv('모델 정확도(논문 기준_fi).csv')
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298
2,SVM,0.998488,0.995298,0.998489,0.996865
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226


In [220]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 100
누적 설명력: 0.9504960804927014


In [221]:
X_test_pca = pca.transform(X_test)

### 랜덤포레스트

In [222]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9418297250062416
평균 카파  : 0.9121642026818


In [223]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train_pca, y_train)
y_pred = rf_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.945141065830721
[[224   3   1]
 [  7 179   9]
 [ 11   4 200]]
                    precision    recall  f1-score   support

           WALKING       0.93      0.98      0.95       228
WALKING_DOWNSTAIRS       0.96      0.92      0.94       195
  WALKING_UPSTAIRS       0.95      0.93      0.94       215

          accuracy                           0.95       638
         macro avg       0.95      0.94      0.94       638
      weighted avg       0.95      0.95      0.94       638

Cohen's Kappa Score: 0.9174


In [224]:
acc_df.loc[0, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [225]:
kappa_df.loc[0, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [226]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.94183,0.945141
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,NaN,NaN
2,SVM,0.998488,0.995298,0.998489,0.996865,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,NaN,NaN


In [227]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,NaN,NaN
2,SVM,0.997721,0.992932,0.997723,0.995288,NaN,NaN
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,NaN,NaN


### 로지스틱회귀

In [228]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', max_iter=1000, random_state=42) # 최적점에 도달 못하여 max_iter=5000로 설정
scores = cross_validate(lr_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9935770588864715
평균 카파  : 0.9903200558321157


In [229]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_pca, y_train)
y_pred = lr_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9968652037617555
[[227   0   1]
 [  0 195   0]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      1.00      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9953


In [230]:
acc_df.loc[1, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [231]:
kappa_df.loc[1, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [232]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865
2,SVM,0.998488,0.995298,0.998489,0.996865,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,NaN,NaN


In [233]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288
2,SVM,0.997721,0.992932,0.997723,0.995288,NaN,NaN
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,NaN,NaN


### SVM

In [234]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9954659913685486
평균 카파  : 0.9931661179936586


In [235]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_pca, y_train)

y_pred = svm_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9968652037617555
[[228   0   0]
 [  0 194   1]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       1.00      1.00      1.00       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9953


In [236]:
acc_df.loc[2, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [237]:
kappa_df.loc[2, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [238]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,NaN,NaN


In [239]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288
2,SVM,0.997721,0.992932,0.997723,0.995288,0.993166,0.995288
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,NaN,NaN


### 결정 트리

In [240]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.847373114099226
평균 카파  : 0.7701628127166479


In [241]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_pca, y_train)

y_pred = dt_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.8322884012539185
[[208  14   6]
 [ 12 165  18]
 [ 20  37 158]]
                    precision    recall  f1-score   support

           WALKING       0.87      0.91      0.89       228
WALKING_DOWNSTAIRS       0.76      0.85      0.80       195
  WALKING_UPSTAIRS       0.87      0.73      0.80       215

          accuracy                           0.83       638
         macro avg       0.83      0.83      0.83       638
      weighted avg       0.84      0.83      0.83       638

Cohen's Kappa Score: 0.7482


In [242]:
acc_df.loc[3, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [243]:
kappa_df.loc[3, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [244]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,0.847373,0.832288


In [245]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288
2,SVM,0.997721,0.992932,0.997723,0.995288,0.993166,0.995288
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,0.770163,0.748165


In [249]:
acc_df.to_csv('모델 정확도(논문 기준_fi_pca).csv', index=False, encoding='utf-8-sig')

In [250]:
kappa_df.to_csv('모델 kappa(논문 기준_fi_pca).csv', index=False, encoding='utf-8-sig')

## 방법3: permutation_importance()

In [251]:
acc_df = pd.read_csv('모델 정확도(논문 기준_fi_pca).csv')
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,0.847373,0.832288


In [292]:
rf_model = joblib.load('rf_model_paper.pkl')
lr_model = joblib.load('lr_model_paper.pkl')
svm_model = joblib.load('svm_model_paper.pkl')
dt_model = joblib.load('dt_model_paper.pkl')

### 랜덤포레스트

In [253]:
model = rf_model

# permutation importance 계산
result_rf = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [254]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result_rf.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 44


In [255]:
importances_df.to_csv('rf_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [256]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9735570852801656
평균 카파  : 0.9601387946860086


In [257]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train_sel, y_train)
y_pred = rf_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9749216300940439
[[221   0   7]
 [  5 189   1]
 [  1   2 212]]
                    precision    recall  f1-score   support

           WALKING       0.97      0.97      0.97       228
WALKING_DOWNSTAIRS       0.99      0.97      0.98       195
  WALKING_UPSTAIRS       0.96      0.99      0.97       215

          accuracy                           0.97       638
         macro avg       0.98      0.97      0.98       638
      weighted avg       0.98      0.97      0.97       638

Cohen's Kappa Score: 0.9623


In [258]:
acc_df.loc[0, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [259]:
kappa_df.loc[0, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [260]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141,0.973557,0.974922
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865,NaN,NaN
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,0.847373,0.832288,NaN,NaN


In [261]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433,0.960139,0.962296
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288,NaN,NaN
2,SVM,0.997721,0.992932,0.997723,0.995288,0.993166,0.995288,NaN,NaN
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,0.770163,0.748165,NaN,NaN


### 로지스틱회귀

In [262]:
model = lr_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [263]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 49


In [264]:
importances_df.to_csv('lr_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [265]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', max_iter=5000, random_state=42)
scores = cross_validate(lr_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9969768520169776
평균 카파  : 0.9954435987207887


In [266]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_sel, y_train)
y_pred = lr_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9952978056426333
[[227   0   1]
 [  0 194   1]
 [  1   0 214]]
                    precision    recall  f1-score   support

           WALKING       1.00      1.00      1.00       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       0.99      1.00      0.99       215

          accuracy                           1.00       638
         macro avg       1.00      1.00      1.00       638
      weighted avg       1.00      1.00      1.00       638

Cohen's Kappa Score: 0.9929


In [267]:
acc_df.loc[1, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [268]:
kappa_df.loc[1, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [269]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141,0.973557,0.974922
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865,0.996977,0.995298
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865,NaN,NaN
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,0.847373,0.832288,NaN,NaN


In [270]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433,0.960139,0.962296
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288,0.995444,0.992932
2,SVM,0.997721,0.992932,0.997723,0.995288,0.993166,0.995288,NaN,NaN
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,0.770163,0.748165,NaN,NaN


### SVM

In [297]:
# permutation importance 계산
result = result_rf  # svm으로 했더니 선택된 피처수가 0개... 랜덤포레스트 것으로 대체

In [298]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 44


In [299]:
importances_df.to_csv('svm_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [300]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9826193957984092
평균 카파  : 0.9737930522974148


In [301]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_sel, y_train)

y_pred = svm_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9858934169278997
[[226   0   2]
 [  1 193   1]
 [  3   2 210]]
                    precision    recall  f1-score   support

           WALKING       0.98      0.99      0.99       228
WALKING_DOWNSTAIRS       0.99      0.99      0.99       195
  WALKING_UPSTAIRS       0.99      0.98      0.98       215

          accuracy                           0.99       638
         macro avg       0.99      0.99      0.99       638
      weighted avg       0.99      0.99      0.99       638

Cohen's Kappa Score: 0.9788


In [302]:
acc_df.loc[2, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [303]:
kappa_df.loc[2, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [304]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141,0.973557,0.974922
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865,0.996977,0.995298
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865,0.982619,0.985893
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,0.847373,0.832288,0.920663,0.913793


In [305]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433,0.960139,0.962296
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288,0.995444,0.992932
2,SVM,0.997721,0.992932,0.997723,0.995288,0.993166,0.995288,0.973793,0.978795
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,0.770163,0.748165,0.880500,0.870631


### 결정트리

In [280]:
model = dt_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [281]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 11


In [282]:
importances_df.to_csv('dt_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [283]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9206634090665906
평균 카파  : 0.8805001913142005


In [284]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_sel, y_train)

y_pred = dt_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9137931034482759
[[208   9  11]
 [  3 184   8]
 [  7  17 191]]
                    precision    recall  f1-score   support

           WALKING       0.95      0.91      0.93       228
WALKING_DOWNSTAIRS       0.88      0.94      0.91       195
  WALKING_UPSTAIRS       0.91      0.89      0.90       215

          accuracy                           0.91       638
         macro avg       0.91      0.91      0.91       638
      weighted avg       0.92      0.91      0.91       638

Cohen's Kappa Score: 0.8706


In [285]:
acc_df.loc[3, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [286]:
kappa_df.loc[3, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [287]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.971667,0.976489,0.972046,0.987461,0.941830,0.945141,0.973557,0.974922
1,LogisticRegression,0.996976,0.998433,0.996977,0.995298,0.993577,0.996865,0.996977,0.995298
2,SVM,0.998488,0.995298,0.998489,0.996865,0.995466,0.996865,0.982619,0.985893
3,DecisionTreeClassifier,0.910849,0.912226,0.910848,0.912226,0.847373,0.832288,0.920663,0.913793


In [288]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.957292,0.964649,0.957869,0.981153,0.912164,0.917433,0.960139,0.962296
1,LogisticRegression,0.995442,0.997644,0.995443,0.992931,0.990320,0.995288,0.995444,0.992932
2,SVM,0.997721,0.992932,0.997723,0.995288,0.993166,0.995288,0.973793,0.978795
3,DecisionTreeClassifier,0.865765,0.868324,0.865748,0.868308,0.770163,0.748165,0.880500,0.870631


In [289]:
acc_df.to_csv('모델 정확도(논문 기준_fi_pca_pi).csv', index=False, encoding='utf-8-sig')

In [290]:
kappa_df.to_csv('모델 kappa(논문 기준_fi_pca_pi).csv', index=False, encoding='utf-8-sig')